# PostgreSQL — Referência Rápida

## Sobre
- **PostgreSQL**: SGBD relacional principal (servidor)
- **postgres**: usuário padrão com role `superuser`
- **psql**: cliente CLI para interagir com o servidor PostgreSQL

## Comandos psql
| Comando | Descrição |
|---|---|
| `\du` | Lista roles e usuários |
| `\dt` | Lista tabelas do schema atual |
| `\d` | Lista tabelas, views e sequences |
| `\l` | Lista bancos de dados |
| `\c <database>` | Conecta a um banco de dados |
| `\d <table_name>` | Descreve a estrutura da tabela |
| `\conninfo` | Exibe detalhes da conexão atual |

# Pipeline ETL — Ações NVDA e AMZN

## Objetivo
Construir um pipeline completo de ETL que ingere dados de ações da bolsa e os armazena no PostgreSQL.

## Etapas
1. **Extract** — Baixar dados de ações via `yfinance`
2. **Transform** — Extrair e transformar com `pandas` (`MultiIndex`, `.xs()`, `reset_index`)
3. **Validate** — Validar registros com modelos `Pydantic`
4. **Load** — Persistir os dados no PostgreSQL via `SQLAlchemy` ORM

---
## Etapa 1 — Extração de Dados

In [ ]:

#imports
import yfinance
import pandas as pd
import json
from sqlalchemy import create_engine, Column, VARCHAR, DATE, FLOAT,INTEGER
from sqlalchemy.orm import declarative_base, sessionmaker
from pydantic import SecretStr, Field, BaseModel, field_validator, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict
from datetime import datetime
from typing import List


In [ ]:
dt_stocks = yfinance.download("NVDA AMZN", period="1mo", interval="1d")

#print(dt_stocks.columns)  # show column names
#print(dt_stocks.dtypes)   # show column data types
#print(dt_stocks.size)     # show total number of elements (rows × columns, including null values)
#print(dt_stocks.count())  # show the number of non-null values per column
#print(dt_stocks)

## MultiIndex com `.xs()`

O `yfinance.download()` retorna um DataFrame com **MultiIndex nas colunas**:
- **Level 0 (Price)**: tipo de dado → `Close`, `High`, `Low`, `Open`, `Volume`
- **Level 1 (Ticker)**: ação → `NVDA`, `AMZN`

O método `.xs()` seleciona uma "fatia" do MultiIndex de forma eficiente, sem filtros complexos:

| Parâmetro | Valores | Descrição |
|---|---|---|
| `axis` | `0` = linhas / `1` = colunas | Eixo de operação |
| `level` | `0` = Price / `1` = Ticker | Nível do MultiIndex a selecionar |

## Notas de Transformação

- **`reset_index()`** — Converte o índice de data em coluna `Date` (de index → coluna)
- **`.filter()`** — Usado apenas para busca por nome de coluna ou índice (não filtra linhas)

In [ ]:
#NVDA STOCKS
nvda_stocks = dt_stocks.xs('NVDA', axis=1, level=1)

#without TICKET level and date index

nvda_stocks = nvda_stocks.reset_index()

filter_nvda = nvda_stocks.Date.max()
max_date_nvda_stocks = nvda_stocks[nvda_stocks.Date==filter_nvda]

#date format -> max_date_nvda_stocks.iloc[0].Date.strftime("%d/%m/%Y")

In [ ]:
#AMZN STOCKS
amzn_stocks = dt_stocks.xs('AMZN', axis=1, level=1)

amzn_stocks = amzn_stocks.reset_index()

#filter
amzn_filter = amzn_stocks.Date.max()
max_date_amzn_stocks = amzn_stocks[amzn_stocks.Date==amzn_filter]

#date format -> max_date_amzn_stocks.iloc[0].Date.strftime("%d/%m/%Y")

---
## Etapa 2 — Validação com Pydantic

In [ ]:
class StockPrice(BaseModel):
    date: datetime = Field(alias="Date")
    close: float = Field(alias="Close")
    high: float = Field(alias="High")
    low: float = Field(alias="Low")
    open: float = Field(alias="Open")
    volume: int = Field(alias="Volume")
    
    @model_validator(mode="before")
    @classmethod
    def validate_blank_spaces(cls, values):
        list_values = ["Date","Close","High","Low","Open","Volume"]
        try:
            for r in list_values:
                if "" or " " in values[r]:
                    values[r] = values[r].strip()
                else:
                    return values
        except Exception as e:
            return e

---
## Etapa 3 — Banco de Dados

In [ ]:
# Environment variables
class Environment(BaseSettings):
    
    model_config = SettingsConfigDict(env_file=".env_0001_postgresql_alchemy")
    
    db_user: SecretStr = Field(..., description="Username for connecting to the database", alias="DATABASE_USER")
    db_password: SecretStr = Field(..., description="Password for the database user", alias="DATABASE_PASSWORD")
    db_name: SecretStr = Field(..., description="Name of the database to connect to", alias="DATABASE_NAME")
    db_address: SecretStr = Field(..., description="Host or IP address of the database",alias="DATABASE_ADDRESS")
    
env = Environment()

In [ ]:
#Database Connection
engine = create_engine(f"postgresql+psycopg://{env.db_user.get_secret_value()}:{env.db_password.get_secret_value()}@{env.db_address.get_secret_value()}/{env.db_name.get_secret_value()}")
Base = declarative_base()

try:
    test_connection = engine.connect()
except Exception as e:
    print(e)

In [ ]:
#database session
database_session = sessionmaker(engine)

---
## Etapa 4 — Modelos SQLAlchemy (ORM)

In [ ]:
class amzn_price(Base):
    
    __tablename__ = "AMZN_PRICE"
    
    id = Column(INTEGER, primary_key=True, autoincrement=True)
    date = Column(VARCHAR(length=10))
    close = Column(FLOAT)
    high = Column(FLOAT)
    low = Column(FLOAT)
    open = Column(FLOAT)
    volume = Column(INTEGER)
    
Base.metadata.create_all(engine)
    

In [ ]:
class nvda_price(Base):
    
    __tablename__ = "NVDA_PRICE"
    
    id = Column(INTEGER, primary_key=True, autoincrement=True)
    date = Column(VARCHAR(length=10))
    close = Column(FLOAT)
    high = Column(FLOAT)
    low = Column(FLOAT)
    open = Column(FLOAT)
    volume = Column(FLOAT)

Base.metadata.create_all(engine)

---
## Etapa 5 — Carregamento dos Dados

In [ ]:
#.iloc -> row x column

amzn_stock_rows = amzn_stocks.shape[0]

with database_session() as session:
    for r in range(amzn_stock_rows):
        json_amzn_stock = amzn_stocks.iloc[r].to_json(date_format="iso")
        amznStockPrice = StockPrice.model_validate_json(json_amzn_stock)
        amzn_price_obj = amzn_price(
            date=amznStockPrice.date.strftime("%d/%m/%Y"),
            close=amznStockPrice.close,
            high=amznStockPrice.high,
            low=amznStockPrice.low,
            open=amznStockPrice.open,
            volume=amznStockPrice.volume,
        )
        session.add(amzn_price_obj)
    session.commit()

    print(session.query(amzn_price).count())

In [ ]:
nvda_stock_rows = nvda_stocks.shape[0]

with database_session() as session:
    for r in range(nvda_stock_rows):
        json_nvda_stock = nvda_stocks.iloc[r].to_json(date_format="iso")
        nvdaStockPrice = StockPrice.model_validate_json(json_nvda_stock)    
        nvda_price_obj = nvda_price(
            date = nvdaStockPrice.date.strftime("%d/%m/%Y"),
            close = nvdaStockPrice.close,
            high = nvdaStockPrice.high,
            low = nvdaStockPrice.low,
            open = nvdaStockPrice.open,
            volume = nvdaStockPrice.volume
        )
        session.add(nvda_price_obj)
    session.commit()
    print(session.query(nvda_price).count())